In [128]:
# Import libraries and load datasets
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import sqlite3
from wordcloud import WordCloud
from datetime import datetime #only need date, not time

df_choc = pd.read_csv('../Data/chocolate_bar_ratings_2022_cleaned.csv')
df_coffee = pd.read_csv('../Data/simplified_coffee_cleaned.csv')

In [115]:
df_choc.head()

,REF,Company,Company Location,Review Date,Bean Origin,Bar Name,Cocoa %,Ingredients,Characteristics,Rating
0,2454,5150,U.S.A.,2019,Tanzania,"Kokoa Kamili, batch 1",76.0,"3- B,S,C","rich cocoa, fatty, bready",3.25
1,2454,5150,U.S.A.,2019,Madagascar,"Bejofo Estate, batch 1",76.0,"3- B,S,C","cocoa, blackberry, full body",3.75
2,2458,5150,U.S.A.,2019,Dominican Republic,"Zorzal, batch 1",76.0,"3- B,S,C","cocoa, vegetal, savory",3.50
3,2542,5150,U.S.A.,2021,Fiji,"Matasawalevu, batch 1",68.0,"3- B,S,C","chewy, off, rubbery",3.00
4,2542,5150,U.S.A.,2021,India,"Anamalai, batch 1",68.0,"3- B,S,C","milk brownie, macadamia,chewy",3.50


In [116]:
df_coffee.head()

,name,roaster,roast,loc_country,origin,100g_USD,rating,review_date,review
0,Ethiopia Shakiso Mormora,Revel Coffee,Medium-Light,United States,Ethiopia,4.70,92,November 2017,"Crisply sweet, cocoa-toned. Lemon blossom, roa..."
1,Ethiopia Suke Quto,Roast House,Medium-Light,United States,Ethiopia,4.19,92,November 2017,"Delicate, sweetly spice-toned. Pink peppercorn..."
2,Ethiopia Gedeb Halo Beriti,Big Creek Coffee Roasters,Medium,United States,Ethiopia,4.85,94,November 2017,"Deeply sweet, subtly pungent. Honey, pear, tan..."
3,Ethiopia Kayon Mountain,Red Rooster Coffee Roaster,Light,United States,Ethiopia,5.14,93,November 2017,"Delicate, richly and sweetly tart. Dried hibis..."
4,Ethiopia Gelgelu Natural Organic,Willoughby's Coffee & Tea,Medium-Light,United States,Ethiopia,3.97,93,November 2017,"High-toned, floral. Dried apricot, magnolia, a..."


I decided not to merge these two DataFrames. Instead, I will change the column names in the chocolate DataFrame to match the coffee DataFrame.  I will also drop the REF column in the chocolate DataFrame because it is not relevant to this analysis.

In [121]:
df_choc.columns

Index(['Company', 'Company Location', 'Review Date', 'Bean Origin', 'Bar Name',
       'Cocoa %', 'Ingredients', 'Characteristics', 'Rating'],
      dtype='str')

In [122]:
df_coffee.columns

Index(['name', 'roaster', 'roast', 'loc_country', 'origin', '100g_USD',
       'rating', 'review_date', 'review'],
      dtype='str')

In [124]:
# rename columns in chocolate dataset
new_df_choc = df_choc.rename(columns={'Company': 'company', 'Company Location': 'loc_country', 'Review Date': 'review_date', 'Bean Origin': 'origin', 'Bar Name': 'bar_name', 'Cocoa %': 'cocoa_%', 'Ingredients': 'ingredients', 'Characteristics': 'review', 'Rating': 'rating'})
new_df_choc.columns

Index(['company', 'loc_country', 'review_date', 'origin', 'bar_name',
       'cocoa_%', 'ingredients', 'review', 'rating'],
      dtype='str')

In [133]:
new_df_choc.columns

Index(['company', 'loc_country', 'review_date', 'origin', 'bar_name',
       'cocoa_%', 'ingredients', 'review', 'rating'],
      dtype='str')

In [135]:
df_coffee.columns

Index(['name', 'roaster', 'roast', 'loc_country', 'origin', '100g_USD',
       'rating', 'review_date', 'review'],
      dtype='str')

After getting a 'not in index' error, I need to rename some coffee columns.  

In [136]:
new_df_coffee = df_coffee.rename(columns={'name': 'coffee_name', 'roaster': 'company'})

In [137]:
new_df_coffee.columns

Index(['coffee_name', 'company', 'roast', 'loc_country', 'origin', '100g_USD',
       'rating', 'review_date', 'review'],
      dtype='str')

I created an ERD with four tables. I will now create a database.

In [139]:
conn = sqlite3.connect('../Data/choc_coffee1.db')

cc_choc_bar_df = new_df_choc[['origin', 'bar_name', 'cocoa_%', 'ingredients', 'review', 'rating']].drop_duplicates()

cc_coffee_df = new_df_coffee[['rating', 'origin', 'coffee_name', 'roast', 'review']].drop_duplicates()

cc_choc_country_df = new_df_choc[['company', 'loc_country', 'rating']].drop_duplicates()

cc_coffee_country_df = new_df_coffee[['rating', 'company', 'loc_country']].drop_duplicates()

cc_choc_bar_df.to_sql('cc_choc_bar', conn, index = False, if_exists = 'replace')
cc_coffee_df.to_sql('cc_coffee', conn, index = False, if_exists = 'replace')
cc_choc_country_df.to_sql('cc_choc_country', conn, index = False, if_exists = 'replace')
cc_coffee_country_df.to_sql('cc_coffee_country', conn, index = False, if_exists = 'replace')

conn.commit

<function Connection.commit()>

I received a KeyError: "['bar_name', 'cocoa_%', 'ingredients'] not in index".  I will check the column names again.  

In [112]:
df_choc.columns

Index(['REF', 'company', 'loc_country', 'review_date', 'origin', 'bar_name',
       'cocoa_%', 'ingredients', 'review', 'rating'],
      dtype='str')

In [110]:
df_coffee.columns

Index(['name', 'roaster', 'roast', 'loc_country', 'origin', '100g_USD',
       'rating', 'review_date', 'review'],
      dtype='str')

Python did not retain my changes to the column names.  I believe that this is the source of the error. Therefore, I will go back to the beginning and delete almost all of the cells.  